# 第9章 面向对象与科研管线

对应正文前置能力：读懂科研 Python 库里的对象接口，并用很轻量的类把数据、状态和方法组织起来。

这一章不追求完整的面向对象理论。Part 0 只需要你掌握三件事：对象是什么、方法怎么调用、为什么很多科研库会写出 `read()`、`fit()`、`predict()` 这样的接口。

## 学习目标

- 理解类、实例、属性和方法的基本含义。
- 能读懂 `object.method()` 这样的接口调用方式。
- 知道如何把一组相关数据和操作封装成一个小对象。
- 认识 `read`、`fit`、`predict` 这类后续常见 API 命名。

## 对象、属性和方法

对象可以理解成“带着数据和行为的值”。例如一颗行星不只是一个名字，它还可以带有半长轴、周期、质量和半径，并能自己计算一些派生量。

In [ ]:
class Planet:
    def __init__(self, name, semi_major_axis_au, orbital_period_years, mass_earth, radius_earth):
        self.name = name
        self.semi_major_axis_au = semi_major_axis_au
        self.orbital_period_years = orbital_period_years
        self.mass_earth = mass_earth
        self.radius_earth = radius_earth

    def is_giant(self):
        return self.mass_earth >= 10.0

    def kepler_ratio(self):
        return self.orbital_period_years ** 2 / self.semi_major_axis_au ** 3

    def describe(self):
        return {
            "name": self.name,
            "giant": self.is_giant(),
            "kepler_ratio": round(self.kepler_ratio(), 3),
        }

    def __repr__(self):
        return f"Planet(name={self.name!r}, period={self.orbital_period_years:.3f} yr)"

## 实例化与方法调用

类只是模板，实例才是具体对象。`self` 指向当前这个实例。后续你看到 `table.read()`、`model.fit()`、`ax.scatter()` 时，都可以先问自己：谁是对象，谁是方法，方法在处理什么状态？

In [ ]:
earth = Planet("Earth", 1.0, 1.0, 1.0, 1.0)
jupiter = Planet("Jupiter", 5.203, 11.862, 317.8, 11.209)

print(earth)
print(jupiter.describe())
print("Jupiter is giant?", jupiter.is_giant())

## 把一组对象打包成一个小目录

科研工作里，常见对象并不是一颗行星，而是一整个目录、表格、样本集或者图像集合。我们可以把这类数据组织成一个小型容器对象，让它自己负责读取、筛选和汇总。

In [ ]:
import csv
from pathlib import Path


class PlanetCatalog:
    def __init__(self, planets):
        self.planets = list(planets)

    @classmethod
    def read(cls, path):
        with path.open(newline="", encoding="utf-8") as handle:
            rows = list(csv.DictReader(handle))
        planets = [
            Planet(
                row["planet"],
                float(row["semi_major_axis_au"]),
                float(row["orbital_period_years"]),
                float(row["mass_earth"]),
                float(row["radius_earth"]),
            )
            for row in rows
        ]
        return cls(planets)

    def __len__(self):
        return len(self.planets)

    def names(self):
        return [planet.name for planet in self.planets]

    def filter_inner(self, limit=2.0):
        return PlanetCatalog([planet for planet in self.planets if planet.semi_major_axis_au <= limit])

    def summary(self):
        periods = [planet.orbital_period_years for planet in self.planets]
        giant_count = sum(1 for planet in self.planets if planet.is_giant())
        return {
            "count": len(self.planets),
            "giant_count": giant_count,
            "mean_period": round(sum(periods) / len(periods), 3),
        }

    def __repr__(self):
        return f"PlanetCatalog(n_planets={len(self)})"

## 读取接口和容器接口

这一类接口最值得先修阶段熟悉的地方，是命名非常稳定：`read()` 往往表示“从磁盘读入对象”，`fit()` 往往表示“用数据更新模型状态”，`predict()` 往往表示“基于已学到的状态给出输出”。你不用一开始就会写它们，但你要能读懂它们。

In [ ]:
catalog = PlanetCatalog.read(Path("../../data/small/planetary_orbits_demo.csv"))
print(catalog)
print("名称列表:", catalog.names())
print("摘要:", catalog.summary())

inner_catalog = catalog.filter_inner(limit=2.0)
print("内侧样本:", inner_catalog.names())

## 一个极小的 `fit` / `predict` 示例

后续机器学习章节会大量出现 `model.fit(...)` 和 `model.predict(...)`。为了先建立直觉，我们用开普勒关系做一个非常小的拟合例子：先从样本中学出比例常数，再用这个常数预测轨道周期。

In [ ]:
class KeplerModel:
    def __init__(self):
        self.constant = None

    def fit(self, planets):
        ratios = [planet.kepler_ratio() for planet in planets]
        self.constant = sum(ratios) / len(ratios)
        return self

    def predict(self, semi_major_axes):
        if self.constant is None:
            raise RuntimeError("先调用 fit() 再调用 predict()")
        return [(self.constant * axis ** 3) ** 0.5 for axis in semi_major_axes]


model = KeplerModel().fit(catalog.planets)
print("拟合常数:", round(model.constant, 3))
print("对前四个半长轴的预测:", [round(value, 3) for value in model.predict([1.0, 1.524, 5.203, 9.537])])

## 如何阅读后续库接口

看到 `table.read()`、`model.fit(X, y)`、`ax.scatter(x, y)` 这类写法时，最有效的阅读顺序是：先找对象，再找方法名，再看参数，最后看返回值。这样你就不会把它们当成一串陌生符号。

## 练习建议

1. 给 `Planet` 增加一个 `density_proxy()` 方法，返回 `mass_earth / radius_earth**3`。
2. 给 `PlanetCatalog` 增加一个 `giant_planets()` 方法。
3. 把 `KeplerModel.predict()` 改成接受一颗 `Planet` 对象列表。
4. 试着把这几个类抄进一个 `.py` 文件，并用 `import` 再读一次。

## 小结

面向对象的核心不是“炫技”，而是把相关状态和操作绑在一起。你不需要先变成 OOP 专家，只要能看懂对象、属性和方法，后续的科学 Python 库就不会显得神秘。